In [0]:
import os
from pyspark.sql import functions as F

BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
BRONZE_PARQUET_DIR = os.path.join(BASE_DIR, "bronze_output", "parquet_data_hhs")
os.makedirs(BRONZE_PARQUET_DIR, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"BRONZE_PARQUET_DIR: {BRONZE_PARQUET_DIR}")

In [0]:
# Reading parquet & create dataframe. 

from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os
filepath_benefits = os.path.join(BRONZE_PARQUET_DIR, 'benefits')

filepath_rates = os.path.join(BRONZE_PARQUET_DIR, 'rates')

def read_parquet(filepath: str) -> DataFrame:
    data_f = spark.read.parquet(filepath)
    return data_f
    
df_benefits = read_parquet(filepath_benefits)
df_rates = read_parquet(filepath_rates)


In [0]:
df_rates.describe()

In [0]:
df_benefits.describe()

In [0]:
display(df_benefits.select("BenefitName").distinct().orderBy("BenefitName"))

In [0]:
display(df_benefits.select("IsCovered").distinct())

In [0]:
df_covered = df_benefits.filter(F.col("IsCovered") == "Covered")

In [0]:
df_grouped = (
    df_covered
    .groupBy("PlanId")
    .agg(
        F.concat_ws(",", F.collect_list("BenefitName")).alias("BenefitNames"),
        F.size(F.collect_list("BenefitName")).alias("BenefitCount")
    )
)

display(df_grouped)

In [0]:
df_benefits_grouped_transformed = (
    df_grouped
    .withColumn("PlanId", F.split(F.col("PlanId"), "-")[0])
    .dropDuplicates(["PlanId"])
)

display(df_grouped_transformed)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def filter_baseline_rate(df: DataFrame) -> DataFrame:
    return (
        df
        .filter(
            (F.col("Age") == "30") &
            (F.col("Tobacco") == "No Preference")
        )
        .select(
            F.col("PlanId"),
            F.col("IndividualRate").cast("double").alias("IndividualRate")
        )
        .filter(F.col("IndividualRate").isNotNull())
        # .dropDuplicates(["PlanId"])
    )


df_rate_baseline = df_rates.transform(filter_baseline_rate)

display(df_rate_baseline)

In [0]:
SILVER_PARQUET_DIR = os.path.join(BASE_DIR, "silver_output", "parquet_data_hhs")
os.makedirs(SILVER_PARQUET_DIR, exist_ok=True)

df_benefits_grouped_transformed.write.mode("overwrite").parquet(os.path.join(SILVER_PARQUET_DIR, "benefits_grouped_transformed"))
df_rate_baseline.write.mode("overwrite").parquet(os.path.join(SILVER_PARQUET_DIR, "rate_baseline"))

# Unit Tests

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

# --- Test filter_baseline_rate ---
def test_filter_baseline_rate_keeps_only_age30_no_preference():
    data = [
        Row(PlanId="PLAN001", Age="30", Tobacco="No Preference", IndividualRate="150.50"),
        Row(PlanId="PLAN002", Age="25", Tobacco="No Preference", IndividualRate="200.00"),
        Row(PlanId="PLAN003", Age="30", Tobacco="Tobacco User", IndividualRate="300.00"),
        Row(PlanId="PLAN004", Age="30", Tobacco="No Preference", IndividualRate=None),
    ]
    df = spark.createDataFrame(data)
    result = df.transform(filter_baseline_rate)
    
    assert result.count() == 1, f"Expected 1 row, got {result.count()}"
    row = result.first()
    assert row["PlanId"] == "PLAN001"
    assert row["IndividualRate"] == 150.50
    print("PASS: test_filter_baseline_rate_keeps_only_age30_no_preference")


# --- Test IsCovered filter ---
def test_filter_covered_benefits():
    data = [
        Row(BenefitName="Dental", IsCovered="Covered", PlanId="P1"),
        Row(BenefitName="Vision", IsCovered="Not Covered", PlanId="P1"),
        Row(BenefitName="Mental Health", IsCovered="Covered", PlanId="P2"),
    ]
    df = spark.createDataFrame(data)
    result = df.filter(F.col("IsCovered") == "Covered")
    
    assert result.count() == 2, f"Expected 2 rows, got {result.count()}"
    names = [r["BenefitName"] for r in result.collect()]
    assert "Dental" in names
    assert "Mental Health" in names
    assert "Vision" not in names
    print("PASS: test_filter_covered_benefits")


# --- Test groupBy aggregation ---
def test_group_benefits_by_plan():
    data = [
        Row(BenefitName="Dental", PlanId="P1"),
        Row(BenefitName="Vision", PlanId="P1"),
        Row(BenefitName="Mental Health", PlanId="P2"),
    ]
    df = spark.createDataFrame(data)
    result = (
        df.groupBy("PlanId")
        .agg(
            F.concat_ws(",", F.collect_list("BenefitName")).alias("BenefitNames"),
            F.size(F.collect_list("BenefitName")).alias("BenefitCount"),
        )
    )
    
    assert result.count() == 2, f"Expected 2 plans, got {result.count()}"
    p1 = result.filter(F.col("PlanId") == "P1").first()
    assert p1["BenefitCount"] == 2
    assert "Dental" in p1["BenefitNames"]
    assert "Vision" in p1["BenefitNames"]
    print("PASS: test_group_benefits_by_plan")


# --- Test PlanId split transformation ---
def test_plan_id_split_takes_first_segment():
    data = [
        Row(PlanId="10064IN005-0001", BenefitNames="Dental", BenefitCount=1),
        Row(PlanId="10064IN005-0002", BenefitNames="Vision", BenefitCount=1),
        Row(PlanId="99999XX001-0001", BenefitNames="Mental", BenefitCount=1),
    ]
    df = spark.createDataFrame(data)
    result = (
        df.withColumn("PlanId", F.split(F.col("PlanId"), "-")[0])
        .dropDuplicates(["PlanId"])
    )
    
    # Two duplicates on "10064IN005" should collapse to one
    assert result.count() == 2, f"Expected 2 rows after dedup, got {result.count()}"
    plan_ids = [r["PlanId"] for r in result.collect()]
    assert "10064IN005" in plan_ids
    assert "99999XX001" in plan_ids
    print("PASS: test_plan_id_split_takes_first_segment")




In [0]:
# --- Run all tests ---
test_filter_baseline_rate_keeps_only_age30_no_preference()
test_filter_covered_benefits()
test_group_benefits_by_plan()
test_plan_id_split_takes_first_segment()
print("\nAll tests passed!")